# 02 — Causal Normalization V2, Processed EDA & Export

**CPU-only.** Đọc dataset YOLO17 và split đã khóa từ Notebook 01. Tạo pose13 `[T,13,4]` cho mỗi clip, trực quan hóa kết quả và xuất ZIP portable cho MLP/GRU. **Không chạy YOLO, không thay raw dataset.**

## Cell 1 — Chọn split từ Notebook 01
Giữ nguyên `RAW_ZIP`. `TEST_PEOPLE` phải giống danh sách đã chạy ở Notebook 01; output theo từng experiment. Chỉ mount Drive và import code chung.

In [ ]:
from pathlib import Path
import os, sys, json, zipfile, hashlib, shutil, tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from IPython.display import display

# Mount Drive trước khi xác định ROOT (tránh lỗi lần chạy đầu ở Colab).
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

# Colab: dùng đúng root Drive; local: tự tìm project root từ thư mục chạy notebook.
COLAB_ROOT = Path('/content/drive/MyDrive/KI_8/IMP302/LAB_2/truongedu')
ROOT = Path(os.environ['POSEFORGE_ROOT']).expanduser() if os.environ.get('POSEFORGE_ROOT') else (
    COLAB_ROOT if COLAB_ROOT.is_dir() else next(
        (p for p in (Path.cwd(), *Path.cwd().parents)
         if (p / 'notebooks').is_dir() and (p / 'shared').is_dir()), Path.cwd()
    )
)
ROOT = ROOT.resolve()
RAW_ZIP = Path(os.environ.get('POSEFORGE_ZIP', str(ROOT / 'raw_dataset_zip/Real_Posture_Dataset_YOLO11n_singleperson_v2.zip')))
TEST_PEOPLE = ['TRUONG']  # PHẢI giống Cell 1 của Notebook 01
SPLIT_SEED = 302
SPLIT_TAG = 'test_' + ('AUTO_seed' + str(SPLIT_SEED) if not TEST_PEOPLE else '_'.join(sorted(TEST_PEOPLE)))
RAW_REPORT = Path(os.environ.get('POSEFORGE_RAW_REPORT', str(ROOT / 'outputs/raw_eda/experiments' / SPLIT_TAG)))
OUT = Path(os.environ.get('POSEFORGE_PROCESSED_OUT', str(ROOT / 'outputs/poseforge_pose13_v2' / SPLIT_TAG)))

# Đặt ROOT (local/Colab/Kaggle) vào Python path để import package shared.
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from shared.dataset_pipeline import file_sha256, COCO13
from shared.pose_features import CausalPoseNormalizer, FEATURE_VERSION, FEATURE_SHAPE, CONFIDENCE_THRESHOLD

assert RAW_ZIP.is_file(), f'Missing raw ZIP: {RAW_ZIP}'
assert (RAW_REPORT / 'split_manifest.json').is_file(), 'Run Notebook 01 with this TEST_PEOPLE first'
assert (RAW_REPORT / 'raw_catalog.csv').is_file(), 'Missing Notebook 01 catalog'
split_info = json.loads((RAW_REPORT / 'split_manifest.json').read_text(encoding='utf-8'))
assert split_info['test_people'] == (sorted(TEST_PEOPLE) if TEST_PEOPLE else split_info['test_people'])
assert split_info['seed'] == SPLIT_SEED
catalog = pd.read_csv(RAW_REPORT / 'raw_catalog.csv')
split_df = pd.read_csv(RAW_REPORT / 'split_manifest.csv')
assert catalog.clip_id.is_unique and split_df.clip_id.is_unique
assert set(catalog.clip_id) == set(split_df.clip_id)
assert set(split_df.loc[split_df.split == 'test','person_id']).isdisjoint(
    set(split_df.loc[split_df.split == 'train_pool','person_id']))
print('Input:', RAW_ZIP)
print('Split:', split_info['test_people'], '| train pool:', len(split_info['train_pool_people']))
print('Normalizer:', FEATURE_VERSION, '| features per frame: 13 × 4 = 52')

## Cell 2 — Sao chép raw ZIP vào CPU runtime, xác minh SHA
Thao tác chỉ đọc, không giải nén MP4. Kiểm tra nguồn đúng phiên bản đã khóa trong Notebook 01.

In [ ]:
LOCAL_ZIP = (Path('/content/poseforge_raw_176clips.zip') if Path('/content').exists()
             else Path(tempfile.gettempdir()) / 'poseforge_raw_phase2.zip')
source_sha = file_sha256(RAW_ZIP)
assert source_sha == split_info['source_zip_sha256'], 'Raw ZIP differs from Notebook 01; stop (split/source mismatch).'
if not LOCAL_ZIP.exists() or LOCAL_ZIP.stat().st_size != RAW_ZIP.stat().st_size or file_sha256(LOCAL_ZIP) != source_sha:
    with RAW_ZIP.open('rb') as src, LOCAL_ZIP.open('wb') as dst, tqdm(total=RAW_ZIP.stat().st_size, unit='B', unit_scale=True, desc='Stage raw ZIP') as p:
        while (chunk := src.read(4 * 1024 * 1024)):
            dst.write(chunk); p.update(len(chunk))
assert file_sha256(LOCAL_ZIP) == source_sha
print('Raw SHA256 verified:', source_sha)
print('Ready to process:', len(catalog), 'camera clips')

## Cell 3 — Đọc YOLO17 → Causal Normalize 13×4 theo clip
Mỗi clip có một normalizer mới, duyệt frame từ đầu đến cuối, không dùng tương lai. Lưu NPZ chứa cả pose13, raw17, frame index, relative timestamp và scale/center để dùng cho MLP hoặc GRU sau này. **Không loại frame ở bước này.**

In [ ]:
FEATURES = OUT / 'features'
FEATURES.mkdir(parents=True, exist_ok=True)
clip_rows, frame_rows = [], []
with zipfile.ZipFile(LOCAL_ZIP) as zf:
    prefix = next(n[:-len('labels.csv')] for n in zf.namelist()
                  if n.endswith('Real_Posture_Dataset/labels.csv'))
    split_lookup = split_df.set_index('clip_id')['split'].to_dict()
    for row in tqdm(catalog.itertuples(index=False), total=len(catalog), desc='Normalize clips', unit='clip'):
        normalizer = CausalPoseNormalizer()
        raw17s, features, centers, scales, indexes, times, widths, heights = [], [], [], [], [], [], [], []
        with zf.open(prefix + row.raw_member) as src:
            for line in src:
                if not line.strip():
                    continue
                data = json.loads(line)
                k = (np.asarray(data['keypoints'], dtype=np.float32) if data.get('keypoints') is not None else None)
                if k is not None:
                    assert k.shape == (17,3) and np.isfinite(k).all(), f'Bad raw17 in {row.clip_id}'
                width, height = int(data['width']), int(data['height'])
                x, center, scale = normalizer.transform(k, width, height, backend='yolo')
                frame_idx = int(data['frame_index'])
                assert frame_idx == len(indexes), f'Frame gap in {row.clip_id}'
                raw17s.append(k if k is not None else np.zeros((17,3), dtype=np.float32))
                features.append(x)
                centers.append(center); scales.append(scale)
                indexes.append(frame_idx)
                times.append(float(data['relative_ms']) if data.get('relative_ms') is not None else np.nan)
                widths.append(width); heights.append(height)
                frame_rows.append(dict(clip_id=row.clip_id, person_id=row.person_id, camera_id=row.camera_id,
                                       label=int(row.label), split=split_lookup[row.clip_id], frame_index=frame_idx,
                                       valid13_count=int(x[:,3].sum()), scale_px=float(scale)))
        assert len(features) == int(row.frames), f'Frame count mismatch in {row.clip_id}'
        pose = np.stack(features).astype(np.float32)
        assert pose.shape == (int(row.frames),13,4) and np.isfinite(pose).all()
        feature_file = FEATURES / f'{row.clip_id}.npz'
        np.savez_compressed(feature_file,
                            x=pose, raw17=np.stack(raw17s).astype(np.float32),
                            frame_index=np.asarray(indexes, dtype=np.int32),
                            relative_ms=np.asarray(times, dtype=np.float64),
                            center_px=np.stack(centers).astype(np.float32),
                            scale_px=np.asarray(scales, dtype=np.float32),
                            width=np.asarray(widths, dtype=np.int32),
                            height=np.asarray(heights, dtype=np.int32))
        clip_rows.append(dict(clip_id=row.clip_id, person_id=row.person_id, label=int(row.label),
                              camera_id=row.camera_id, session_id=row.session_id, take_id=row.take_id,
                              split=split_lookup[row.clip_id], frames=len(features),
                              feature_path=f'features/{row.clip_id}.npz'))
index_df = pd.DataFrame(clip_rows)
frame_df = pd.DataFrame(frame_rows)
print('Processed:', len(index_df), 'clips /', len(frame_df), 'frames; .npz files:', len(list(FEATURES.glob('*.npz'))))
display(index_df.head())

## Cell 4 — Quality checks & split validation
Kiểm tra shape, NaN/Inf, mask/confidence, ID và số clip/frame. `train_pool`/`test` tách theo người; validation chưa được chọn. Không buộc đủ 13/13 điểm để tránh bỏ L2/L4.

In [ ]:
assert len(index_df) == len(catalog)
assert len(frame_df) == int(catalog.frames.sum())
assert (frame_df.valid13_count.between(0,13)).all()
assert frame_df.clip_id.nunique() == len(index_df)
assert not (set(index_df[index_df.split == 'test'].person_id) &
            set(index_df[index_df.split == 'train_pool'].person_id))
assert set(index_df.label) == set(range(6))
# Spot-check all saved arrays, no feature clipping except validity/confidence from shared normalizer.
for row in tqdm(index_df.itertuples(index=False), total=len(index_df), desc='Validate NPZ', unit='clip'):
    with np.load(OUT / row.feature_path, allow_pickle=False) as z:
        x = z['x']; raw = z['raw17']; idx = z['frame_index']
        assert x.shape == (row.frames, 13, 4) and raw.shape == (row.frames, 17, 3)
        assert np.isfinite(x).all() and np.array_equal(idx, np.arange(row.frames))
        assert np.isin(x[:,:,3], [0.,1.]).all()
        assert ((x[:,:,2]>=0)&(x[:,:,2]<=1)).all()
        assert np.all(x[:,:,0:2][x[:,:,3]==0] == 0)
print('NPZ and leakage checks PASSED')
print('Frames with all 13 visible:', int((frame_df.valid13_count==13).sum()))
print('Frames partial:', int(frame_df.valid13_count.between(1,12).sum()))
print('Frames with no usable upper-body joint:', int((frame_df.valid13_count==0).sum()))
display(frame_df.groupby(['split','label']).agg(frames=('frame_index','size'),
                       mean_valid13=('valid13_count','mean')).reset_index())

## Cell 5 — Test causal consistency: offline = frame-by-frame, không có future leak
Chạy lại từ raw17 theo đúng thứ tự frame và so với NPZ, sau đó kiểm tra thêm frame tương lai không làm thay đổi các frame trước.

In [ ]:
for row in index_df.head(min(4, len(index_df))).itertuples(index=False):
    with np.load(OUT / row.feature_path, allow_pickle=False) as z:
        raw, saved = z['raw17'], z['x']
        widths, heights = z['width'], z['height']
        normalizer = CausalPoseNormalizer()
        streamed = np.stack([normalizer.transform(k, int(w), int(h))[0]
                             for k,w,h in zip(raw,widths,heights)])
        np.testing.assert_allclose(saved, streamed, atol=1e-6, rtol=0)
        m = min(15, len(raw))
        normalizer = CausalPoseNormalizer()
        prefix_result = np.stack([normalizer.transform(k, int(w), int(h))[0]
                                   for k,w,h in zip(raw[:m], widths[:m], heights[:m])])
        np.testing.assert_allclose(saved[:m], prefix_result, atol=1e-6, rtol=0)
print('Causal consistency PASSED on 4 sample clips (including prefix/no-future test)')

## Cell 6 — Processed EDA: hiển thị và lưu biểu đồ
Thống kê từ **train_pool** để tránh dùng test phân phối cho việc chọn đặc trưng/tham số. Có biểu đồ số frame theo lớp, số joints hợp lệ, scale, normalized x/y, coverage camera và skeleton chuẩn hóa.

In [ ]:
CHARTS = OUT / 'reports/charts'
CHARTS.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'figure.dpi': 100, 'savefig.dpi': 130, 'font.size': 10})
def show_save(fig, name):
    fig.tight_layout(); fig.savefig(CHARTS / name, bbox_inches='tight'); plt.show(); plt.close(fig)
train = frame_df[frame_df.split == 'train_pool']
fig, ax = plt.subplots(figsize=(8,3.5))
counts = train.label.value_counts().reindex(range(6), fill_value=0)
ax.bar([f'L{i}' for i in range(6)], counts.values)
ax.set(xlabel='Label', ylabel='Frames', title='Train-pool frames by label')
show_save(fig, '01_train_frames_by_label.png')

fig, ax = plt.subplots(figsize=(8,3.5))
valid = train.valid13_count.value_counts().reindex(range(14), fill_value=0)
ax.bar(valid.index, valid.values)
ax.set(xlabel='Usable upper-body joints (0–13)', ylabel='Frames', title='Train-pool pose quality', xticks=range(14))
show_save(fig, '02_valid_joint_histogram.png')

fig, ax = plt.subplots(figsize=(7.5,3.5))
for cam, group in train.groupby('camera_id'):
    ax.hist(group.scale_px, bins=25, alpha=.6, label=cam)
ax.legend(); ax.set(xlabel='Causal scale (px)', ylabel='Frames', title='Causal scale distribution by camera')
show_save(fig, '03_scale_by_camera.png')

sample_xy=[]
for row in index_df[index_df.split=='train_pool'].head(24).itertuples(index=False):
    with np.load(OUT / row.feature_path, allow_pickle=False) as z:
        xy = z['x'][:,:,:2]; mask = z['x'][:,:,3].astype(bool)
        sample_xy.append(xy[mask])
points = np.concatenate(sample_xy,axis=0) if sample_xy else np.empty((0,2))
fig, ax = plt.subplots(figsize=(7.5,3.5))
for col,label in [(0,'x_norm'),(1,'y_norm')]:
    ax.hist(np.clip(points[:,col], -5,5),bins=45,alpha=.6,label=label)
ax.legend();ax.set(xlabel='Relative coordinate (clipped for display at ±5)',ylabel='Keypoints',title='Normalized XY — TRAIN only')
show_save(fig, '04_normalized_xy_hist.png')

fig, ax = plt.subplots(figsize=(7.5,3.5))
coverage = train.groupby('camera_id').valid13_count.mean() / 13 * 100
ax.bar(coverage.index, coverage.values);ax.set(ylabel='Mean % visible joints',ylim=(0,100),title='Train-pool joint coverage by camera')
show_save(fig, '05_joint_coverage_camera.png')

# One skeleton from train only: compare raw pixel coordinates vs causal xy.
first = index_df[index_df.split=='train_pool'].iloc[0]
with np.load(OUT / first.feature_path,allow_pickle=False) as z:
    raw, processed = z['raw17'][0,:13], z['x'][0]
mask = processed[:,3].astype(bool)
EDGES = [(0,1),(0,2),(1,3),(2,4),(5,6),(5,7),(7,9),(6,8),(8,10),(5,11),(6,12),(11,12)]
fig, axes = plt.subplots(1,2,figsize=(10,4))
for ax, coords, title in [(axes[0],raw[:,:2],'Raw pixel XY'), (axes[1],processed[:,:2],'Causal normalized XY')]:
    for a,b in EDGES:
        if mask[a] and mask[b]: ax.plot(coords[[a,b],0],coords[[a,b],1],color='steelblue')
    ax.scatter(coords[mask,0],coords[mask,1],s=25,color='darkorange')
    ax.set(title=title,aspect='equal');ax.invert_yaxis()
show_save(fig,'06_raw_vs_normalized_skeleton.png')

## Cell 7 — Export processed NPZ + reports + ZIP portable
Lưu index/split/feature contract, checksums và ZIP **chỉ chứa các NPZ nhỏ + CSV/JSON/biểu đồ**. ZIP review riêng không chứa NPZ. Paths tương đối; dùng được trên Colab/Kaggle/local.

In [ ]:
from datetime import datetime, timezone

index_df.to_csv(OUT / 'index.csv', index=False)
frame_df.to_csv(OUT / 'frame_quality.csv.gz', index=False, compression='gzip')
split_df.to_csv(OUT / 'split_manifest.csv', index=False)
(OUT / 'split_manifest.json').write_text(json.dumps(split_info, ensure_ascii=False, indent=2))
contract = dict(version=FEATURE_VERSION, extractor='yolo11n-pose.pt',
                joints=COCO13, channels=['x_norm','y_norm','confidence','valid_mask'],
                shape_per_frame=[13,4], flatten_dim=52,
                confidence_threshold=CONFIDENCE_THRESHOLD, ema_alpha=0.25,
                normalization='center shoulders when visible, causal EMA scale',
                state_scope='one clip/person/camera; reset per clip',
                missing_policy='zero xy/conf and mask=0; never require all 13 visible',
                split_strategy='subject_wise_multi_test',
                source_zip_sha256=source_sha)
(OUT / 'feature_contract.json').write_text(json.dumps(contract, ensure_ascii=False, indent=2))
summary = dict(clips=len(index_df), frames=len(frame_df), test_people=split_info['test_people'],
               train_pool_people=split_info['train_pool_people'],
               pose13_full13=int((frame_df.valid13_count==13).sum()),
               partial13=int(frame_df.valid13_count.between(1,12).sum()),
               no_valid13=int((frame_df.valid13_count==0).sum()),
               source_zip_sha256=source_sha)
(OUT / 'processed_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2))
checksums = []
for p in sorted(FEATURES.glob('*.npz')):
    checksums.append(f'{file_sha256(p)}  {p.relative_to(OUT)}')
(OUT / 'checksums.sha256').write_text('\n'.join(checksums) + '\n')
assert len(checksums) == len(index_df)

processed_zip = OUT.parent / f'poseforge_pose13_v2_{SPLIT_TAG}.zip'
review_zip = OUT.parent / f'processed_eda_review_{SPLIT_TAG}.zip'
with zipfile.ZipFile(processed_zip, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for p in sorted(OUT.rglob('*')):
        if p.is_file(): archive.write(p,arcname=str(Path('poseforge_pose13_v2') / p.relative_to(OUT)))
with zipfile.ZipFile(review_zip, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for p in sorted(OUT.rglob('*')):
        if p.is_file() and 'features' not in p.parts and p.suffix != '.npz':
            archive.write(p,arcname=str(Path('poseforge_pose13_v2') / p.relative_to(OUT)))
with zipfile.ZipFile(processed_zip) as z:
    assert z.testzip() is None
print('PROCESSED READY:', processed_zip, f'({processed_zip.stat().st_size / 1024**2:.2f} MB)')
print('REVIEW ZIP:', review_zip, f'({review_zip.stat().st_size / 1024**2:.2f} MB)')
print('Charts:', len(list(CHARTS.glob('*.png'))))
print('Output: per-clip x[T,13,4], flatten x.reshape(T,52) for MLP; windows later for GRU.')